In [1]:
import pandas as pd
import duckdb

In [2]:
df_uber = pd.read_parquet("../Drafts/df_uber_parquet.parquet")

In [3]:
df_uber.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'pickup_longitude',
       'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag',
       'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount',
       'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
       'improvement_surcharge', 'total_amount'],
      dtype='object')

In [4]:
dim_date_pick_up = pd.read_parquet("../Final/Dim_Date_Pick_Up.parquet")
dim_date_drop_off = pd.read_parquet("../Final/Dim_Date_Drop_Off.parquet")
dim_pick_up_location = pd.read_parquet("../Final/Dim_PickUp_Location.parquet")
dim_drop_off_location = pd.read_parquet("../Final/Dim_DropOff_Location.parquet")
dim_travel = pd.read_parquet("../Final/Dim_Travel.parquet")
dim_rate = pd.read_parquet("../Final/Dim_Rate.parquet")
dim_payment = pd.read_parquet("../Final/Dim_Payment.parquet")

Column Time Taken Creation

In [5]:
df_uber = duckdb.sql(
    """
        SELECT *, tpep_dropoff_datetime - tpep_pickup_datetime as time_taken
        FROM df_uber
    """
)

JOINING FOR TIME ID FOR BOTH PICKUP AND DROPOFF

In [6]:
dim_date_pick_up.columns

Index(['pick_up_time_id', 'date_time', 'day', 'hour', 'month', 'year',
       'weekday', 'is_weekend'],
      dtype='object')

In [7]:
dim_date_drop_off.columns

Index(['drop_off_time_id', 'date_time', 'day', 'hour', 'month', 'year',
       'weekday', 'is_weekend'],
      dtype='object')

In [8]:
df_uber.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'pickup_longitude',
 'pickup_latitude',
 'RatecodeID',
 'store_and_fwd_flag',
 'dropoff_longitude',
 'dropoff_latitude',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'time_taken']

In [9]:
df_uber = duckdb.sql(
    """
        SELECT u.*,p.pick_up_time_id,d.drop_off_time_id
        FROM df_uber u
        LEFT JOIN dim_date_pick_up p
        ON u.tpep_pickup_datetime = p.date_time
        LEFT JOIN dim_date_drop_off d
        ON u.tpep_dropoff_datetime = d.date_time
    """
)

In [10]:
df_uber.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'pickup_longitude',
 'pickup_latitude',
 'RatecodeID',
 'store_and_fwd_flag',
 'dropoff_longitude',
 'dropoff_latitude',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'time_taken',
 'pick_up_time_id',
 'drop_off_time_id']

JOINING FOR TRAVEL ID

In [11]:
dim_travel.columns

Index(['travel_id', 'pick_up_id', 'drop_off_id'], dtype='object')

In [12]:
df_uber = duckdb.sql(
    """
        SELECT u.*, t.travel_id
        FROM df_uber u
        LEFT JOIN dim_travel t
        ON u.pick_up_time_id = t.pick_up_id
        AND u.drop_off_time_id = t.drop_off_id
    """
)

JOINING FOR LOCATION ID

In [13]:
dim_pick_up_location.columns

Index(['pickup_location_key', 'pickup_longitude', 'pickup_latitude'], dtype='object')

In [14]:
df_uber = duckdb.sql(
    """
        SELECT u.*, p.pickup_location_key
        FROM df_uber u
        LEFT JOIN dim_pick_up_location p
        ON u.pickup_longitude = p.pickup_longitude
        AND u.pickup_latitude = p.pickup_latitude
    """
)

In [15]:
dim_drop_off_location.columns 

Index(['dropoff_location_key', 'dropoff_longitude', 'dropoff_latitude'], dtype='object')

In [16]:
df_uber = duckdb.sql(
    """
        SELECT u.*, d.dropoff_location_key
        FROM df_uber u
        LEFT JOIN dim_drop_off_location d
        ON u.dropoff_longitude = d.dropoff_longitude
        AND u.dropoff_latitude = d.dropoff_latitude
    """
)

In [17]:
df_uber.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'pickup_longitude',
 'pickup_latitude',
 'RatecodeID',
 'store_and_fwd_flag',
 'dropoff_longitude',
 'dropoff_latitude',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'time_taken',
 'pick_up_time_id',
 'drop_off_time_id',
 'travel_id',
 'pickup_location_key',
 'dropoff_location_key']

CREATING FINAL FACT TABLE

In [18]:
df_uber = df_uber.to_df()

Fact_Trips = df_uber[['VendorID','travel_id','time_taken','passenger_count',
 'pickup_location_key','dropoff_location_key','RatecodeID','fare_amount',
 'extra','mta_tax','tip_amount','tolls_amount','improvement_surcharge',
 'total_amount','payment_type']].copy()

In [19]:
Fact_Trips.dtypes

VendorID                           int64
travel_id                          int64
time_taken               timedelta64[us]
passenger_count                    int64
pickup_location_key                int64
dropoff_location_key               int64
RatecodeID                         int64
fare_amount                      float64
extra                            float64
mta_tax                          float64
tip_amount                       float64
tolls_amount                     float64
improvement_surcharge            float64
total_amount                     float64
payment_type                       int64
dtype: object

In [20]:
Fact_Trips.rename(columns={
    'VendorID' : 'vendor_id',
    'pickup_location_key' : 'pick_up_location',
    'dropoff_location_key' : 'dropp_off_location',
    'RatecodeID' : 'rate_code',
})

,vendor_id,travel_id,time_taken,passenger_count,pick_up_location,dropp_off_location,rate_code,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,payment_type
0,1,1,0 days 00:07:55,1,1,1,1,9.0,0.5,0.5,2.05,0.00,0.3,12.35,1
1,1,2,0 days 00:11:06,1,2,2,1,11.0,0.5,0.5,3.05,0.00,0.3,15.35,1
2,2,3,0 days 00:31:06,2,3,3,1,54.5,0.5,0.5,8.00,0.00,0.3,63.80,1
3,2,7,0 days 00:00:00,3,4,4,1,31.5,0.0,0.5,3.78,5.54,0.3,41.62,1
4,2,7,0 days 00:00:00,5,5,5,3,98.0,0.0,0.0,0.00,15.50,0.3,113.80,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100297,2,94467,0 days 00:03:58,5,5197,93001,1,4.5,0.0,0.5,0.00,0.00,0.3,5.30,2
100298,2,94467,0 days 00:03:58,5,5197,93001,1,-4.5,0.0,-0.5,0.00,0.00,-0.3,-5.30,3
100299,2,94467,0 days 00:03:58,1,92666,93005,1,4.5,0.0,0.5,0.00,0.00,0.3,5.30,2
100300,2,97191,0 days 00:01:04,5,95315,95665,1,3.0,0.0,0.5,0.00,0.00,0.3,3.80,2


In [21]:
Fact_Trips.to_parquet("../Final/Fact_Trips.parquet",index=False)